In [ ]:
import glob, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['axes.unicode_minus'] = False

DATA_DIR = os.path.dirname(os.path.abspath('plot.ipynb'))

# Select CSV: use env var TITA_CSV, or latest *_result.csv
csv_path = os.environ.get('TITA_CSV', None)
if csv_path is None:
    files = sorted(glob.glob(os.path.join(DATA_DIR, '*_result.csv')))
    if not files:
        raise FileNotFoundError('No *_result.csv found.')
    csv_path = files[-1]

csv_name = os.path.basename(csv_path)
ts       = csv_name.replace('_result.csv', '')
IMG_DIR  = DATA_DIR

print(f'CSV      : {csv_name}')
df = pd.read_csv(csv_path, sep=';')
t  = df['time'].values
print(f'Duration : {t[-1]-t[0]:.1f}s  |  Samples: {len(df):,}')

MODE_COLORS = {'INIT':'#9e9e9e','GO':'#4caf50','BACK':'#f44336','LEFT':'#2196f3','RIGHT':'#ff9800'}

def draw_mode_bg(ax):
    prev, s = df['mode'].iloc[0], t[0]
    for i in range(1, len(df)):
        cur = df['mode'].iloc[i]
        if cur != prev or i == len(df)-1:
            ax.axvspan(s, t[i], alpha=0.12, color=MODE_COLORS.get(prev, '#fff'))
            s, prev = t[i], cur

def save(fig, tag):
    p = os.path.join(IMG_DIR, f'{ts}_result_{tag}.png')
    fig.savefig(p, dpi=120, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved: {p}')
    return p

saved_images = {}

In [ ]:
# ── MPC State vs Reference ────────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(13, 11), sharex=True)
fig.suptitle(f'MPC State vs Reference  [{ts}]', fontsize=13)
[draw_mode_bg(ax) for ax in axes]

axes[0].plot(t, df['x_int'],   label='x_int',   color='steelblue')
axes[0].plot(t, df['xRef_x'],  label='xRef_x',  color='steelblue', ls='--', alpha=0.7)
axes[0].set_ylabel('m'); axes[0].set_title('Position x'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(t, np.degrees(df['pitch']),      label='pitch',   color='tomato')
axes[1].plot(t, np.degrees(df['xRef_theta']), label='xRef_th', color='tomato', ls='--', alpha=0.7)
axes[1].set_ylabel('deg'); axes[1].set_title('Pitch theta'); axes[1].legend(); axes[1].grid(True)

axes[2].plot(t, df['vel'],     label='vel',     color='seagreen')
axes[2].plot(t, df['xRef_dx'], label='xRef_dx', color='seagreen', ls='--', alpha=0.7)
axes[2].set_ylabel('m/s'); axes[2].set_title('Forward velocity'); axes[2].legend(); axes[2].grid(True)

axes[3].plot(t, df['ang_vel_y'], color='purple', label='ang_vel_y')
axes[3].set_ylabel('rad/s'); axes[3].set_title('Pitch rate'); axes[3].legend(); axes[3].grid(True)
axes[3].set_xlabel('Time (s)')

fig.tight_layout()
saved_images['mpc_state'] = save(fig, 'mpc_state')

In [ ]:
# ── Control Output (Torque) ───────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
fig.suptitle(f'Control Output (Torque)  [{ts}]', fontsize=13)
[draw_mode_bg(ax) for ax in axes]

axes[0].plot(t, df['u_cmd'], color='navy', label='u_cmd')
axes[0].set_ylabel('Nm'); axes[0].set_title('u_cmd (common torque)'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(t, df['u_L'], color='darkorange', label='u_L')
axes[1].plot(t, df['u_R'], color='dodgerblue',  label='u_R')
axes[1].set_ylabel('Nm'); axes[1].set_title('L/R final torque'); axes[1].legend(); axes[1].grid(True)

axes[2].plot(t, df['u_L'] - df['u_R'], color='crimson', label='u_L - u_R')
axes[2].set_ylabel('Nm'); axes[2].set_title('Yaw torque diff'); axes[2].legend(); axes[2].grid(True)
axes[2].set_xlabel('Time (s)')

fig.tight_layout()
saved_images['torque'] = save(fig, 'torque')

In [ ]:
# ── IMU Orientation ───────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
fig.suptitle(f'IMU Orientation  [{ts}]', fontsize=13)
[draw_mode_bg(ax) for ax in axes]

axes[0].plot(t, np.degrees(df['roll']),      color='r', label='roll')
axes[0].set_ylabel('deg'); axes[0].set_title('Roll'); axes[0].legend(); axes[0].grid(True)
axes[1].plot(t, np.degrees(df['pitch_imu']), color='g', label='pitch')
axes[1].set_ylabel('deg'); axes[1].set_title('Pitch'); axes[1].legend(); axes[1].grid(True)
axes[2].plot(t, np.degrees(df['yaw']),       color='b', label='yaw')
axes[2].set_ylabel('deg'); axes[2].set_title('Yaw'); axes[2].legend(); axes[2].grid(True)
axes[2].set_xlabel('Time (s)')

fig.tight_layout()
saved_images['imu'] = save(fig, 'imu')

In [ ]:
# ── Mode Timeline ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 3))
fig.suptitle(f'Mode Timeline  [{ts}]', fontsize=13)

prev, s = df['mode'].iloc[0], t[0]
for i in range(1, len(df)):
    cur = df['mode'].iloc[i]
    if cur != prev or i == len(df)-1:
        ax.axvspan(s, t[i], alpha=0.5, color=MODE_COLORS.get(prev, '#fff'))
        s, prev = t[i], cur

ax.plot(t, df['pitch'], color='black', lw=0.8, label='pitch (rad)')
patches = [mpatches.Patch(color=c, label=m) for m, c in MODE_COLORS.items()]
ax.legend(handles=patches, loc='upper right', fontsize=8)
ax.set_xlabel('Time (s)'); ax.set_ylabel('pitch (rad)'); ax.grid(True)

fig.tight_layout()
saved_images['mode'] = save(fig, 'mode')

print('\n=== Generated Images ===')
for k, v in saved_images.items():
    print(f'  {k:12s}: {os.path.basename(v)}')